# <span style="color: #1F1DB5;">SPRINT 18 – Deep Learning y Visión Artificial

# <span style="color: #1F1DB5;">Notebook 2 – Aprendizaje No Supervisado: Clustering y PCA — Versión estudiante

## Cómo usar esta versión estudiante

Este notebook está preparado para que tomes apuntes, completes ejercicios y documentes tus decisiones durante la clase.

**Modo de trabajo recomendado:**

1. Lee primero la consigna de cada bloque o ejercicio.
2. Completa los espacios de respuesta antes de comparar con la explicación del instructor/a.
3. Ejecuta las celdas de setup cuando existan.
4. Escribe interpretación, dudas y decisiones en los espacios de apuntes.
5. Al final, revisa si tus respuestas conectan datos, método e implicación de negocio.

> Las salidas de ejecución fueron limpiadas para que puedas correr el notebook desde cero.


## Notas de clase principales

- Diferenciar entre aprendizaje supervisado y no supervisado.
- Aplicar K-Means para segmentación de datos paso a paso.
- Determinar el número óptimo de clusters con elbow method y silhouette score.
- Aplicar clustering jerárquico y leer dendrogramas.
- Usar PCA para visualización de datos de alta dimensión.

### Mis notas iniciales

- 
- 
- 


## <span style="color: #2826DE;">Objetivos de Aprendizaje

- Diferenciar entre **aprendizaje supervisado y no supervisado**.
- Aplicar **K-Means** para segmentación de datos paso a paso.
- Determinar el número óptimo de clusters con **elbow method y silhouette score**.
- Aplicar **clustering jerárquico** y leer dendrogramas.
- Usar **PCA** para visualización de datos de alta dimensión.

### <span style="color: #2772F2;">Agenda (2 horas)

| Tema | Duración |
|---|---|
Introducción + Pregunta guía | 10 min |
Supervisado vs No Supervisado | 15 min |
K-Means: algoritmo e implementación | 25 min |
Número óptimo de clusters | 20 min |
Clustering jerárquico y PCA | 20 min |
Pair Programming | 15 min |
Cierre y Kahoot | 5 min |

## <span style="color: #2826DE;">❓ Pregunta Guía

### ¿Es posible descubrir patrones en los datos sin tener una respuesta correcta que aprender?

Hasta ahora siempre teníamos una variable objetivo (y): precio, churn, sentimiento. Pero ¿qué pasa cuando solo tienes datos y quieres descubrir patrones ocultos? Ahí entra el aprendizaje no supervisado: dejas que el algoritmo encuentre la estructura por sí solo.

### Mi respuesta inicial

- 


## <span style="color: #2826DE;">Supervisado vs No Supervisado (15 mins)

| Aspecto | Supervisado | No Supervisado |
|---|---|---|
| Datos | Con etiquetas (X, y) | Sin etiquetas (solo X) |
| Objetivo | Predecir y | Descubrir estructura |
| Evaluación | Métricas claras (accuracy, RMSE) | Métricas indirectas (silhouette, inercia) |
| Ejemplos | Clasificación, regresión | Clustering, reducción de dimensión |
| Analogía | Examen con respuestas | Explorar sin mapa |

**Tipos de aprendizaje no supervisado:**

1. **Clustering:** Agrupar datos similares.
   - Casos de uso: segmentación de clientes, agrupación de documentos, detección de anomalías.

2. **Reducción de dimensionalidad:** Comprimir features manteniendo información.
   - Casos de uso: visualización, preprocesamiento, eliminación de ruido.

3. **Detección de anomalías:** Encontrar puntos que no encajan.
   - Casos de uso: fraude, defectos en manufactura, intrusiones.

**¿Cuándo usar no supervisado?**
- No tienes etiquetas (muy común en la industria).
- Quieres explorar y entender tus datos antes de modelar.
- Necesitas segmentar clientes/productos/comportamientos.

Preguntas:

- ¿Puedes pensar en un caso de tu vida donde querrías agrupar cosas sin categorías predefinidas?

- ¿Por qué es más difícil evaluar un modelo no supervisado que uno supervisado?

### Mis respuestas de validación

- 
- 


## <span style="color: #2826DE;">K-Means: Algoritmo e Implementación (25 mins)

**K-Means** es el algoritmo de clustering más popular por su simplicidad y eficiencia.

**Algoritmo paso a paso:**
1. Elige **k** (número de clusters deseados).
2. Coloca **k centroides** al azar en el espacio.
3. **Asigna** cada punto al centroide más cercano (distancia euclídea).
4. **Recalcula** los centroides como la media de sus puntos asignados.
5. **Repite** pasos 3-4 hasta que los centroides no cambien (convergencia).

**Características:**
- Rápido y escalable (O(n × k × iteraciones)).
- Siempre converge (pero puede ser a un óptimo local).
- Asume clusters esféricos de tamaño similar.
- Sensible a la inicialización → usa `init='k-means++'`.

**Limitaciones:**
- Necesitas definir k de antemano.
- No funciona bien con clusters de formas irregulares.
- Sensible a outliers (porque usa la media).

Implementemos K-Means desde la intuición hasta sklearn, usando un dataset de clientes de un centro comercial.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

# Generar datos de clientes (simulación de Mall Customers)
np.random.seed(42)
n = 200

# 5 segmentos naturales de clientes
segments = {
    'Jóvenes bajo gasto': (25, 20, 30, 10),   # edad_media, gasto_medio, std_edad, std_gasto
    'Jóvenes alto gasto': (30, 80, 5, 10),
    'Mediana edad moderado': (45, 50, 8, 15),
    'Mayores bajo gasto': (60, 25, 8, 10),
    'Mayores alto gasto': (55, 75, 7, 12),
}

data = []
for name, (age_m, spend_m, age_s, spend_s) in segments.items():
    n_seg = n // 5
    ages = np.random.normal(age_m, age_s, n_seg).clip(18, 70)
    spending = np.random.normal(spend_m, spend_s, n_seg).clip(1, 100)
    for a, s in zip(ages, spending):
        data.append({'Edad': a, 'Gasto_Anual': s})

df = pd.DataFrame(data)
print(f"Dataset: {df.shape}")
print(df.describe().round(1))

# Visualización antes de clustering
plt.figure(figsize=(8, 6))
plt.scatter(df['Edad'], df['Gasto_Anual'], alpha=0.5, s=30)
plt.xlabel('Edad')
plt.ylabel('Gasto Anual (miles MXN)')
plt.title('Clientes del Centro Comercial - ¿Cuántos grupos ves?')
plt.show()

In [ ]:
# Escalar datos (K-Means es sensible a la escala)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df[['Edad', 'Gasto_Anual']])

# Aplicar K-Means con k=5
kmeans = KMeans(n_clusters=5, init='k-means++', n_init=10, random_state=42)
df['Cluster'] = kmeans.fit_predict(X_scaled)

# Visualización con clusters
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Antes
axes[0].scatter(df['Edad'], df['Gasto_Anual'], alpha=0.5, s=30, color='gray')
axes[0].set_title('Antes de K-Means')
axes[0].set_xlabel('Edad')
axes[0].set_ylabel('Gasto Anual (miles)')

# Después
colors = ['#e41a1c', '#377eb8', '#4daf4a', '#984ea3', '#ff7f00']
for k in range(5):
    mask = df['Cluster'] == k
    axes[1].scatter(df.loc[mask, 'Edad'], df.loc[mask, 'Gasto_Anual'],
                   alpha=0.6, s=30, color=colors[k], label=f'Cluster {k}')
# Centroides (desescalados)
centroids = scaler.inverse_transform(kmeans.cluster_centers_)
axes[1].scatter(centroids[:, 0], centroids[:, 1], marker='X', s=200,
               c='black', edgecolors='white', linewidths=2, label='Centroides')
axes[1].set_title('Después de K-Means (k=5)')
axes[1].set_xlabel('Edad')
axes[1].set_ylabel('Gasto Anual (miles)')
axes[1].legend()

plt.tight_layout()
plt.show()

# Perfil de cada cluster
print("\nPerfil de cada cluster:")
print(df.groupby('Cluster')[['Edad', 'Gasto_Anual']].mean().round(1))

## <span style="color: #2826DE;">Número Óptimo de Clusters (20 mins)

El mayor reto de K-Means: **¿cómo elegir k?** Dos métodos populares:

**1. Método del Codo (Elbow Method):**
- Calcula la **inercia** (suma de distancias al centroide) para k=1, 2, 3...
- Grafica inercia vs k.
- El "codo" (donde la curva deja de bajar rápido) sugiere el k óptimo.
- Limitación: a veces el codo no es claro.

**2. Silhouette Score:**
- Mide qué tan bien asignado está cada punto a su cluster.
- Rango: -1 a 1. Más alto = mejor.
  - Cerca de 1: bien asignado (lejos de otros clusters).
  - Cerca de 0: en la frontera entre clusters.
  - Negativo: probablemente en el cluster equivocado.
- Ventaja: Da un valor único y claro para comparar.

Apliquemos ambos métodos a nuestros datos de clientes para determinar el k óptimo.

In [ ]:
from sklearn.metrics import silhouette_score

# Elbow Method y Silhouette para diferentes k
K_range = range(2, 11)
inertias = []
silhouettes = []

for k in K_range:
    km = KMeans(n_clusters=k, init='k-means++', n_init=10, random_state=42)
    labels = km.fit_predict(X_scaled)
    inertias.append(km.inertia_)
    silhouettes.append(silhouette_score(X_scaled, labels))

# Gráficas
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Elbow
axes[0].plot(K_range, inertias, 'bo-', linewidth=2)
axes[0].axvline(5, color='red', linestyle='--', alpha=0.7, label='k=5 (codo)')
axes[0].set_xlabel('Número de clusters (k)')
axes[0].set_ylabel('Inercia')
axes[0].set_title('Método del Codo')
axes[0].legend()

# Silhouette
axes[1].plot(K_range, silhouettes, 'gs-', linewidth=2)
axes[1].axvline(5, color='red', linestyle='--', alpha=0.7, label=f'k=5 (sil={silhouettes[3]:.3f})')
axes[1].set_xlabel('Número de clusters (k)')
axes[1].set_ylabel('Silhouette Score')
axes[1].set_title('Silhouette Score')
axes[1].legend()

plt.tight_layout()
plt.show()

print(f"Silhouette scores:")
for k, s in zip(K_range, silhouettes):
    marker = " ← MEJOR" if s == max(silhouettes) else ""
    print(f"  k={k}: {s:.4f}{marker}")

Preguntas:

- ¿Por qué el método del codo puede ser ambiguo? ¿Qué hacer cuando no hay un codo claro?

- ¿Qué nos dice un silhouette score de 0.6 vs uno de 0.3?

### Mis respuestas de validación

- 
- 


## <span style="color: #2826DE;">Clustering Jerárquico y PCA (20 mins)

**Clustering Jerárquico (Agglomerative):**
- No necesitas definir k de antemano.
- Construye una jerarquía de clusters de abajo hacia arriba:
  1. Cada punto empieza como su propio cluster.
  2. En cada paso, une los dos clusters más cercanos.
  3. Repite hasta tener un solo cluster.
- El **dendrograma** muestra toda la jerarquía. Tú decides dónde "cortar".

**PCA (Principal Component Analysis):**
- Reduce dimensiones preservando la mayor varianza posible.
- Transforma features correlacionadas en **componentes principales** independientes.
- Uso clave: visualizar datos de alta dimensión en 2D o 3D.
- Ayuda a ver si los clusters están realmente separados.

Apliquemos clustering jerárquico con dendrograma, y luego usemos PCA para visualizar clusters en un espacio con más dimensiones.

In [ ]:
from scipy.cluster.hierarchy import dendrogram, linkage, fcluster
from sklearn.decomposition import PCA

# --- Clustering Jerárquico ---
# Calcular linkage (con subconjunto para velocidad)
sample_idx = np.random.choice(len(X_scaled), 50, replace=False)
X_sample = X_scaled[sample_idx]

Z = linkage(X_sample, method='ward')  # Ward minimiza varianza intra-cluster

# Dendrograma
fig, ax = plt.subplots(figsize=(14, 6))
dendrogram(Z, ax=ax, leaf_rotation=90, leaf_font_size=8,
           color_threshold=4)
ax.axhline(4, color='red', linestyle='--', label='Corte sugerido (5 clusters)')
ax.set_title('Dendrograma - Clustering Jerárquico')
ax.set_xlabel('Muestras')
ax.set_ylabel('Distancia')
ax.legend()
plt.tight_layout()
plt.show()

print("El dendrograma muestra la jerarquía de uniones.")
print("Cortamos donde las ramas son 'largas' (mucha distancia entre uniones).")

In [ ]:
# --- PCA para visualización ---
# Creamos datos con más dimensiones para demostrar PCA
np.random.seed(42)
# Agregamos features extra correlacionadas
df_extended = df[['Edad', 'Gasto_Anual']].copy()
df_extended['Ingreso'] = df['Gasto_Anual'] * 1.5 + np.random.normal(0, 10, len(df))
df_extended['Visitas_Mes'] = df['Gasto_Anual'] / 5 + np.random.normal(0, 3, len(df))
df_extended['Años_Cliente'] = (df['Edad'] - 18) * 0.3 + np.random.normal(0, 2, len(df))

X_ext = StandardScaler().fit_transform(df_extended)

# PCA: 5 dimensiones → 2 dimensiones
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_ext)

print(f"Varianza explicada por cada componente: {pca.explained_variance_ratio_.round(3)}")
print(f"Varianza total explicada: {pca.explained_variance_ratio_.sum():.1%}")

# Visualizar clusters en espacio PCA
fig, ax = plt.subplots(figsize=(10, 7))
for k in range(5):
    mask = df['Cluster'] == k
    ax.scatter(X_pca[mask, 0], X_pca[mask, 1],
              alpha=0.6, s=40, color=colors[k], label=f'Cluster {k}')
ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%} varianza)')
ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%} varianza)')
ax.set_title('Clusters visualizados con PCA (5D → 2D)')
ax.legend()
plt.tight_layout()
plt.show()

Preguntas:

- ¿Cuál es la ventaja del clustering jerárquico sobre K-Means?

- ¿Qué información perdemos al reducir de 5 dimensiones a 2 con PCA?

- ¿Por qué es importante escalar los datos antes de aplicar K-Means o PCA?

### Mis respuestas de validación

- 
- 


## <span style="color: #2826DE;">Actividad práctica – Pair Programming (15 mins)

**Dinámica:** En parejas, driver y navigator. Cambien a la mitad.

**Ejercicio:** Segmentar el dataset de abajo (datos de vinos):
1. Escalar los datos.
2. Determinar el k óptimo con elbow y silhouette.
3. Aplicar K-Means con el k elegido.
4. Usar PCA para visualizar los clusters en 2D.
5. ¿Los clusters corresponden a los tipos de vino?

### Espacio de trabajo del estudiante

**Respuesta, decisiones o interpretación:**

- 
- 


In [ ]:
# TODO estudiante: completa el código de acuerdo con la consigna anterior.
# Contexto: Dataset de vinos (13 features, 3 clases reales)


## <span style="color: #2826DE;">Tips y buenas prácticas

- **Siempre escala** antes de clustering. K-Means usa distancias euclídeas (sensible a escala).
- El método del codo es subjetivo. Complementa con silhouette para decisiones más objetivas.
- K-Means asume clusters esféricos. Si sospechas formas irregulares, prueba DBSCAN.
- PCA es excelente para **exploración visual**, pero no necesariamente para modelado.
- Interpreta los clusters: ¿qué significan en el contexto del negocio? Un cluster sin interpretación no es útil.
- Cuidado con la "maldición de la dimensionalidad": con muchas features, las distancias pierden significado.

## <span style="color: #2826DE;">Cierre y Kahoot (5 mins)

Kahoot - Preguntas sugeridas

1️⃣ ¿Cuál es la principal diferencia entre aprendizaje supervisado y no supervisado?

- La cantidad de datos necesarios
- Si los datos tienen etiquetas o no 
- La velocidad del algoritmo
- El lenguaje de programación usado


2️⃣ ¿Qué criterio usa K-Means para asignar puntos a clusters?

- Similitud coseno
- Correlación de Pearson
- Distancia euclídea al centroide más cercano 
- Frecuencia de aparición


3️⃣ ¿Qué indica el "codo" en el método del codo?

- Que el modelo está sobreajustado
- El k donde agregar más clusters ya no mejora significativamente 
- Que hay un error en los datos
- El punto de convergencia del algoritmo


4️⃣ ¿Qué rango de valores tiene el silhouette score?

- 0 a 100
- -1 a 1 
- 0 a 1
- -infinito a infinito


5️⃣ ¿Qué hace PCA?

- Clasifica datos en categorías
- Reduce dimensiones preservando la mayor varianza posible 
- Predice valores futuros
- Elimina outliers automáticamente


6️⃣ ¿Qué muestra un dendrograma?

- La precisión del modelo
- La jerarquía de uniones en clustering jerárquico 
- La distribución de los datos
- Los pesos de una red neuronal